# Unfolding (PRL Product B Wiener-SVD)

**Step 2** of the Product B data-release unfold.

1. Load response matrices from [`unfolding-prepare.ipynb`](unfolding-prepare.ipynb)
2. Load Product B CategorySummary **`total_xsec`** covariance
3. Compute `xsec_unit` from Gen1 ray-traced flux × `FV_split_truncY` × data POT
4. **Closure test** (Asimov / MC): unfold selected MC reco; compare to `A_c @ model`
5. **Data unfold**; save under `PRL/unfolded/`

Wiener-SVD settings (unchanged): `C_type=2`, `Norm_type=0.0`, `stat_scaling=xsec_unit`.

**Legacy Gen1 recovered-cov path:** [`unfolding-legacy-gen1.ipynb`](unfolding-legacy-gen1.ipynb)

## Physics notes

- Cov recipe (archive `unfolding-data`, **not** Gen1 `CovRotation` recovery):
  `Covariance = cov_from_fraccov(total_xsec_frac, nevts_sel_reco) * xsec_unit**2`
- `measured = (n_data - n_mc_bkg) * xsec_unit`, `model = nevts_allmc * xsec_unit`
- Product B `total_xsec` categories: flux, g4, mcstat, detector, cosmics, **genie_xsec**, pot, ntargets
- Do **not** use Gen1 `mean(model/nevts_allmc)` for `xsec_unit`
- Sep-1 Product B ≠ May Gen1 → χ² vs generators will differ from 34.5/12


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
import json
import shutil
import warnings
from datetime import datetime, timezone
from pathlib import Path
from os import makedirs

import numpy as np
import matplotlib.pyplot as plt

REPO = Path("/exp/sbnd/app/users/munjung/xsec/freeze/cafpyana")
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / "analysis_village/numucc_1p0pi/scripts"))

warnings.filterwarnings("ignore", category=FutureWarning)

from analysis_village.numucc_1p0pi.constants import M_AR, N_A, RHO
from analysis_village.numucc_1p0pi.final_selected_evt_vars import CORE_SELECTED_EVT_VARIABLE_CONFIGS
from analysis_village.numucc_1p0pi.syst_category_summary import (
    load_category_syst_summary,
    total_cov_frac,
)
from analysis_village.numucc_1p0pi.syst_disk_layout import category_summary_npz_path
from analysis_village.numucc_1p0pi.utils import (
    cov_from_fraccov,
    get_chi2,
    get_integrated_flux,
    plot_heatmap,
    plot_unfolded_result,
    fig_ext,
    dpi,
)
from analysis_village.unfolding.wienersvd import WienerSVD
from analysis_village.flux.raytrace_volume_defs import (
    FV_SPLIT_TRUNCY_BOXES,
    RAYTRACE_VOLUME_LABEL,
)
from unfolding_data import pack_unfold_results


In [ ]:
PRL_ROOT = Path("/exp/sbnd/data/users/munjung/xsec/numucc_1p0pi/PRL")
RESP_NPZ = PRL_ROOT / "response_matrices/response_matrices.npz"
SYST_ROOT = PRL_ROOT / "systematics/productB_sel_mup"
CAT_NPZ = Path(category_summary_npz_path(str(SYST_ROOT)))
OUT_DIR = PRL_ROOT / "unfolded"
FIG_DIR = OUT_DIR / "plots"
FLUX_FILE = Path("/exp/sbnd/data/users/munjung/flux/SBND_gsimple_raytrace/Gen1.root")

C_TYPE = 2
NORM_TYPE = 0.0

UNFOLD_VAR_CONFIGS = list(CORE_SELECTED_EVT_VARIABLE_CONFIGS)
vc_by = {vc.var_save_name: vc for vc in UNFOLD_VAR_CONFIGS}

makedirs(OUT_DIR, exist_ok=True)
makedirs(FIG_DIR, exist_ok=True)

print("response :", RESP_NPZ, "exists=", RESP_NPZ.is_file())
print("category :", CAT_NPZ, "exists=", CAT_NPZ.is_file())
print("flux     :", FLUX_FILE, "exists=", FLUX_FILE.is_file())
print("out      :", OUT_DIR)


## 1. Load response pack + Product B `total_xsec`


In [ ]:
blob = np.load(RESP_NPZ, allow_pickle=True)
meta = json.loads(str(blob["meta_json"][0]))
data_tot_pot = float(meta["data_tot_pot"])

variables = {}
# discover var names from keys
vsns = sorted({k.split("::", 1)[0] for k in blob.files if "::" in k})
for vsn in vsns:
    pack = {}
    for key in blob.files:
        if key.startswith(vsn + "::"):
            field = key.split("::", 1)[1]
            pack[field] = blob[key]
    variables[vsn] = pack

print("meta:", json.dumps({k: meta[k] for k in ("schema", "n_evt_good", "data_tot_pot", "mc_pot_scale")}, indent=2))
print("variables:", vsns)

summary = load_category_syst_summary(str(CAT_NPZ))
print("category summary variables:", sorted(summary["by_var"].keys())[:8], "...")


## 2. Cross-section unit (flux × FV_split_truncY × data POT)


In [ ]:
print("Fiducial volume (FV_split_truncY):", RAYTRACE_VOLUME_LABEL["FV_split_truncY"])
v_sbnd = 0.0
for i, box in enumerate(FV_SPLIT_TRUNCY_BOXES):
    dx = box["x_range"][1] - box["x_range"][0]
    dy = box["y_range"][1] - box["y_range"][0]
    dz = box["z_range"][1] - box["z_range"][0]
    v_box = dx * dy * dz
    v_sbnd += v_box
    print(f"  slab {i+1}: -> {v_box:.4e} cm3")
print(f"V_SBND = {v_sbnd:.6e} cm3")

integrated_flux_per_pot = get_integrated_flux(str(FLUX_FILE), plot=False)
integrated_flux = integrated_flux_per_pot * data_tot_pot
n_targets = (RHO * v_sbnd / M_AR) * N_A
XSEC_UNIT = 1.0 / (integrated_flux * n_targets)

flux_info = {
    "flux_file": str(FLUX_FILE),
    "flux_fv": "FV_split_truncY",
    "flux_fv_label": RAYTRACE_VOLUME_LABEL["FV_split_truncY"],
    "volume_cm3": float(v_sbnd),
    "integrated_flux_per_pot": float(integrated_flux_per_pot),
    "data_tot_pot": float(data_tot_pot),
    "integrated_flux": float(integrated_flux),
    "n_targets": float(n_targets),
    "xsec_unit": float(XSEC_UNIT),
}
print(f"integrated flux x POT = {integrated_flux:.6e} /cm2")
print(f"N_targets = {n_targets:.4e}")
print(f"xsec_unit = {XSEC_UNIT:.6e} cm2/nucleon")


## 3. Closure test (Asimov / MC)

Unfold the selected signal reco spectrum as if it were data:
`measured = nevts_sel_reco * xsec_unit`, with MC-stat-only diagonal covariance
(plus a tiny floor). Compare unfolded result to `A_c @ model`.


In [ ]:
def asimov_covariance(nevts_sel_reco, xsec_unit):

    """MC-stat-only cov for closure (diagonal in event counts, scaled to xsec)."""

    n = np.asarray(nevts_sel_reco, dtype=float)

    # fractional MC-stat ~ 1/n with floor

    frac = np.where(n > 0, 1.0 / n, 0.0)

    frac_cov = np.diag(frac)

    return cov_from_fraccov(frac_cov, n) * (xsec_unit ** 2)





closure_results = {}

for vsn in vsns:

    vc = vc_by.get(vsn)

    if vc is None:

        print("skip (no VariableConfig):", vsn)

        continue

    pack = variables[vsn]

    response = np.asarray(pack["response"], dtype=float)

    nevts_allmc = np.asarray(pack["nevts_allmc"], dtype=float)

    nevts_sel_reco = np.asarray(pack["nevts_sel_reco"], dtype=float)

    model = nevts_allmc * XSEC_UNIT

    measured = nevts_sel_reco * XSEC_UNIT

    cov = asimov_covariance(nevts_sel_reco, XSEC_UNIT)



    unfold = WienerSVD(

        response, model, measured, cov,

        C_TYPE, NORM_TYPE, stat_scaling=XSEC_UNIT,

    )

    ac_model = np.asarray(unfold["AddSmear"], dtype=float) @ model

    u = np.asarray(unfold["unfold"], dtype=float)

    ucov = np.asarray(unfold["UnfoldCov"], dtype=float)



    # chi2 vs smeared truth

    mask = (u > 0) & (ac_model > 0)

    ndof = int(np.count_nonzero(mask))

    if ndof >= 1:

        chi2, p_val = get_chi2(u[mask], ac_model[mask], ucov[np.ix_(mask, mask)])

    else:

        chi2, p_val = float("nan"), float("nan")



    print(f"[closure] {vsn}: chi2/ndof = {chi2:.3f}/{ndof}  p={p_val:.3g}")

    plot_unfolded_result(

        unfold,

        measured,

        {"GENIE (smeared truth)": ac_model, "GENIE (truth)": model},

        vc,

        plot=True,

        save_fig=True,

        save_name=str(FIG_DIR / f"{vsn}-closure"),

        closure_test=True,

    )

    closure_results[vsn] = {

        "chi2": float(chi2),

        "ndof": int(ndof),

        "unfold": u,

        "AddSmear": np.asarray(unfold["AddSmear"], dtype=float),

    }



print("closure done")



## 4. Data unfold

`measured = (n_data - n_mc_bkg) * xsec_unit`  
`Covariance = cov_from_fraccov(total_xsec_frac, nevts_sel_reco) * xsec_unit**2`


In [ ]:
data_results = {}
ingredients = {}

for vsn in vsns:
    vc = vc_by.get(vsn)
    if vc is None:
        continue
    pack = variables[vsn]
    response = np.asarray(pack["response"], dtype=float)
    nevts_allmc = np.asarray(pack["nevts_allmc"], dtype=float)
    nevts_sel_reco = np.asarray(pack["nevts_sel_reco"], dtype=float)
    n_sel_data = np.asarray(pack["n_sel_data"], dtype=float)

    model = nevts_allmc * XSEC_UNIT
    measured = n_sel_data * XSEC_UNIT

    try:
        frac_cov = total_cov_frac(summary, vsn, kind="xsec")
    except KeyError as ex:
        print(f"[data] skip {vsn}: no total_xsec ({ex})")
        continue

    Covariance = cov_from_fraccov(frac_cov, nevts_sel_reco) * (XSEC_UNIT ** 2)

    unfold = WienerSVD(
        response, model, measured, Covariance,
        C_TYPE, NORM_TYPE, stat_scaling=XSEC_UNIT,
    )

    ac_model = np.asarray(unfold["AddSmear"], dtype=float) @ model
    u = np.asarray(unfold["unfold"], dtype=float)
    ucov = np.asarray(unfold["UnfoldCov"], dtype=float)
    mask = (u > 0) & (ac_model > 0)
    ndof = int(np.count_nonzero(mask))
    if ndof >= 1:
        chi2, p_val = get_chi2(u[mask], ac_model[mask], ucov[np.ix_(mask, mask)])
    else:
        chi2, p_val = float("nan"), float("nan")

    print(f"[data] {vsn}: chi2/ndof = {chi2:.3f}/{ndof}  p={p_val:.3g}")
    plot_unfolded_result(
        unfold,
        measured,
        {"GENIE AR23": ac_model},
        vc,
        plot=True,
        save_fig=True,
        save_name=str(FIG_DIR / f"{vsn}-data_unfold"),
    )
    if response.shape[0] > 1:
        plot_heatmap(
            np.asarray(unfold["AddSmear"], dtype=float),
            vc.bins,
            plot_labels=["True", "True", f"{vsn} AddSmear (A_c)"],
            plot=True,
            save_fig=True,
            save_name=str(FIG_DIR / f"{vsn}-AddSmear"),
        )

    packed = pack_unfold_results(unfold, vc)
    packed["chi2_vs_genie"] = float(chi2)
    packed["ndof_vs_genie"] = int(ndof)
    packed["measured"] = measured
    packed["model"] = model
    packed["response"] = response
    packed["Covariance_input"] = Covariance
    packed["frac_cov_total_xsec"] = np.asarray(frac_cov, dtype=float)
    packed["n_sel_data"] = n_sel_data
    packed["nevts_sel_reco"] = nevts_sel_reco
    packed["nevts_allmc"] = nevts_allmc
    packed["xsec_unit"] = float(XSEC_UNIT)

    data_results[vsn] = packed
    ingredients[vsn] = {
        "bins": np.asarray(pack["bins"], dtype=float),
        "response": response,
        "eff": np.asarray(pack["eff"], dtype=float),
        "reco_vs_true": np.asarray(pack["reco_vs_true"], dtype=float),
        "n_data": np.asarray(pack["n_data"], dtype=float),
        "n_mc_bkg": np.asarray(pack["n_mc_bkg"], dtype=float),
        "n_sel_data": n_sel_data,
        "nevts_allmc": nevts_allmc,
        "nevts_sel_reco": nevts_sel_reco,
        "measured": measured,
        "model": model,
        "Covariance_input": Covariance,
        "frac_cov_total_xsec": np.asarray(frac_cov, dtype=float),
    }

print("data unfold done for", list(data_results))


## 5. Save flux + unfolded products for data release


In [ ]:
# Copy flux file into release dir + write flux_info.json
flux_dest = OUT_DIR / "Gen1_flux.root"
if FLUX_FILE.is_file():
    if not flux_dest.exists() or flux_dest.stat().st_size != FLUX_FILE.stat().st_size:
        shutil.copy2(FLUX_FILE, flux_dest)
        print("copied flux ->", flux_dest)
    else:
        print("flux already present:", flux_dest)
else:
    print("WARN: flux file missing:", FLUX_FILE)

flux_json = OUT_DIR / "flux_info.json"
with open(flux_json, "w") as f:
    json.dump(flux_info, f, indent=2)
print("wrote", flux_json)

# Combined pickle + flat npz
import pickle

release = {
    "meta": {
        "schema": "prl_productB_unfolded_v1",
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "response_npz": str(RESP_NPZ),
        "category_summary_npz": str(CAT_NPZ),
        "c_type": C_TYPE,
        "norm_type": NORM_TYPE,
        "xsec_unit": float(XSEC_UNIT),
        "data_tot_pot": float(data_tot_pot),
        "flux_info": flux_info,
        "response_meta": meta,
        "closure": {k: {"chi2": v["chi2"], "ndof": v["ndof"]} for k, v in closure_results.items()},
    },
    "ingredients": ingredients,
    "results": data_results,
}

pkl_path = OUT_DIR / "unfolding_ingredients_and_results.pkl"
with open(pkl_path, "wb") as f:
    pickle.dump(release, f, protocol=pickle.HIGHEST_PROTOCOL)
print("wrote", pkl_path)

# Flat NPZ for easy loading without pickle
savez = {
    "meta_json": np.array([json.dumps(release["meta"], default=str)], dtype=object),
    "xsec_unit": np.float64(XSEC_UNIT),
    "data_tot_pot": np.float64(data_tot_pot),
}
for vsn, res in data_results.items():
    for key in (
        "bins", "bin_centers", "bin_widths",
        "unfold", "unfold_per_bin_width",
        "stat_err", "syst_err", "total_err",
        "stat_err_per_bin_width", "syst_err_per_bin_width", "total_err_per_bin_width",
        "AddSmear", "UnfoldCov", "StatUnfoldCov", "SystUnfoldCov",
        "measured", "model", "response", "Covariance_input",
        "frac_cov_total_xsec", "n_sel_data", "nevts_sel_reco", "nevts_allmc",
    ):
        if key in res:
            savez[f"{vsn}::{key}"] = np.asarray(res[key])
    savez[f"{vsn}::chi2_vs_genie"] = np.float64(res.get("chi2_vs_genie", np.nan))
    savez[f"{vsn}::ndof_vs_genie"] = np.int32(res.get("ndof_vs_genie", -1))

npz_path = OUT_DIR / "unfolding_ingredients_and_results.npz"
np.savez_compressed(npz_path, **savez)
print("wrote", npz_path)

# Per-variable slim files
per_var_dir = OUT_DIR / "by_variable"
makedirs(per_var_dir, exist_ok=True)
for vsn, res in data_results.items():
    p = per_var_dir / f"{vsn}.npz"
    np.savez_compressed(
        p,
        bins=res["bins"],
        bin_centers=res["bin_centers"],
        unfold=res["unfold"],
        UnfoldCov=res["UnfoldCov"],
        StatUnfoldCov=res["StatUnfoldCov"],
        SystUnfoldCov=res["SystUnfoldCov"],
        AddSmear=res["AddSmear"],
        measured=res["measured"],
        model=res["model"],
        response=res["response"],
        xsec_unit=np.float64(XSEC_UNIT),
        chi2_vs_genie=np.float64(res.get("chi2_vs_genie", np.nan)),
        ndof_vs_genie=np.int32(res.get("ndof_vs_genie", -1)),
    )
    print("wrote", p)

manifest = {
    "schema": "prl_productB_unfolded_v1",
    "created_utc": release["meta"]["created_utc"],
    "pkl": str(pkl_path),
    "npz": str(npz_path),
    "flux_info": str(flux_json),
    "flux_file_copy": str(flux_dest),
    "variables": sorted(data_results),
    "closure_chi2": {k: {"chi2": v["chi2"], "ndof": v["ndof"]} for k, v in closure_results.items()},
}
with open(OUT_DIR / "unfolded_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)
print("wrote", OUT_DIR / "unfolded_manifest.json")
print("Done.")
